# Module 16: Convolutional Neural Networks (CNNs)

**Goal:** Understand how CNNs exploit spatial structure in images and when to use them.

**Prerequisites:** Module 15 (Neural Networks)

**Expected Runtime:** ~25 minutes

**Outputs:**
- Built and trained a CNN on Fashion-MNIST
- Visualized convolution filters and feature maps
- Compared CNN vs dense network performance

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)

## Part 1: Why Spatial Structure Matters

Images have spatial structure: nearby pixels are related. Dense networks ignore this by flattening the image into a 1D vector.

In [ ]:
# Load Fashion-MNIST
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()

# Class names for Fashion-MNIST
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

print(f"Training set: {X_train.shape[0]} images of {X_train.shape[1]}x{X_train.shape[2]} pixels")
print(f"Test set: {X_test.shape[0]} images")
print(f"Classes: {class_names}")

In [ ]:
# Visualize some examples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(class_names[y_train[i]])
    ax.axis('off')
plt.suptitle('Fashion-MNIST: Clothing Classification', fontsize=14)
plt.tight_layout()
plt.show()

print("\n💡 Each 28x28 image has spatial patterns (edges, shapes, textures).")
print("   A CNN preserves these patterns. A dense network flattens them away.")

In [ ]:
# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Add channel dimension for CNN (28, 28) -> (28, 28, 1)
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

# Flatten for dense network (28, 28) -> (784,)
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"CNN input shape: {X_train_cnn.shape}")
print(f"Dense input shape: {X_train_flat.shape}")

## Part 2: The Convolution Operation

A convolution slides a small filter (kernel) over the image, computing dot products at each position. This detects local patterns like edges and textures.

In [ ]:
def apply_filter(image, kernel, title):
    """Apply a convolution filter to an image and visualize."""
    from scipy.ndimage import convolve
    filtered = convolve(image, kernel)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(kernel, cmap='RdBu', vmin=-1, vmax=1)
    axes[1].set_title(f'Filter: {title}')
    for i in range(kernel.shape[0]):
        for j in range(kernel.shape[1]):
            axes[1].text(j, i, f'{kernel[i,j]:.1f}', ha='center', va='center')
    axes[1].axis('off')
    
    axes[2].imshow(filtered, cmap='gray')
    axes[2].set_title('After Convolution')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Get a sample image
sample_img = X_train[5]  # A sandal

# Edge detection filters
horizontal_edge = np.array([[-1, -1, -1],
                            [ 0,  0,  0],
                            [ 1,  1,  1]])

vertical_edge = np.array([[-1, 0, 1],
                          [-1, 0, 1],
                          [-1, 0, 1]])

sharpen = np.array([[ 0, -1,  0],
                    [-1,  5, -1],
                    [ 0, -1,  0]])

apply_filter(sample_img, horizontal_edge, 'Horizontal Edge')
apply_filter(sample_img, vertical_edge, 'Vertical Edge')
apply_filter(sample_img, sharpen, 'Sharpen')

print("💡 CNNs learn these filters automatically! Early layers learn edges,")
print("   deeper layers learn complex patterns like shapes and textures.")

## Part 3: Pooling and Downsampling

Pooling reduces spatial dimensions while preserving important features. Max pooling takes the maximum value in each region.

In [ ]:
def visualize_pooling(image, pool_size=2):
    """Visualize max pooling effect."""
    h, w = image.shape
    new_h, new_w = h // pool_size, w // pool_size
    
    pooled = np.zeros((new_h, new_w))
    for i in range(new_h):
        for j in range(new_w):
            region = image[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            pooled[i, j] = np.max(region)
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title(f'Original: {image.shape}')
    axes[0].axis('off')
    
    axes[1].imshow(pooled, cmap='gray')
    axes[1].set_title(f'After 2x2 Max Pooling: {pooled.shape}')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return pooled

pooled = visualize_pooling(sample_img)

print(f"Original: {sample_img.shape} = {sample_img.size} values")
print(f"Pooled: {pooled.shape} = {pooled.size} values")
print(f"Reduction: {1 - pooled.size/sample_img.size:.0%} fewer values")
print("\n💡 Pooling reduces computation and adds translation invariance.")
print("   The image is smaller but key features are preserved.")

## Part 4: Build a CNN

Standard CNN architecture: Conv -> Pool -> Conv -> Pool -> Flatten -> Dense

In [ ]:
# Build CNN model
cnn_model = keras.Sequential([
    # First conv block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Second conv block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Third conv block
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    # Classification head
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("=== CNN Architecture ===")
cnn_model.summary()

In [ ]:
# Visualize the architecture
print("\n=== Layer-by-Layer Breakdown ===")
print("")
print("Input:        (28, 28, 1)    -- 28x28 grayscale image")
print("Conv2D(32):   (26, 26, 32)   -- 32 filters learn 32 different patterns")
print("MaxPool:      (13, 13, 32)   -- Halve spatial dimensions")
print("Conv2D(64):   (11, 11, 64)   -- 64 filters on smaller features")
print("MaxPool:      (5, 5, 64)     -- Halve again")
print("Conv2D(64):   (3, 3, 64)     -- Final conv layer")
print("Flatten:      (576,)         -- 3*3*64 = 576 features")
print("Dense(64):    (64,)          -- Fully connected")
print("Dense(10):    (10,)          -- 10 class probabilities")

## Part 5: Train and Evaluate

In [ ]:
# Train CNN (use subset for speed in Colab)
print("Training CNN...")
history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
test_loss, test_acc = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\n=== Test Results ===")
print(f"Test Accuracy: {test_acc:.1%}")
print(f"Test Loss: {test_loss:.4f}")

# Per-class performance
y_pred = cnn_model.predict(X_test_cnn, verbose=0).argmax(axis=1)
print("\n=== Per-Class Performance ===")
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(10), class_names, rotation=45, ha='right')
plt.yticks(range(10), class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

# Add text annotations
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.show()

print("💡 Common confusions: Shirt vs T-shirt, Pullover vs Coat")
print("   These are visually similar categories!")

## Part 6: Visualize Learned Filters and Feature Maps

In [ ]:
# Get first conv layer weights
conv1_weights = cnn_model.layers[0].get_weights()[0]  # Shape: (3, 3, 1, 32)

# Visualize first 16 filters
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < 32:
        ax.imshow(conv1_weights[:, :, 0, i], cmap='RdBu', vmin=-0.5, vmax=0.5)
    ax.axis('off')

plt.suptitle('First Conv Layer: 32 Learned Filters (3x3)', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 These filters were learned automatically during training.")
print("   Some detect edges, others detect textures and patterns.")

In [ ]:
# Visualize feature maps for a sample image
from tensorflow.keras.models import Model
import tensorflow as tf

try:
    _ = cnn_model(tf.zeros((1, 28, 28, 1)))

    layer_outputs = [layer.output for layer in cnn_model.layers[:5]]
    activation_model = Model(inputs=cnn_model.input, outputs=layer_outputs)

    sample = X_test_cnn[0:1]
    activations = activation_model.predict(sample, verbose=0)

    first_conv_activation = activations[0][0]

    fig, axes = plt.subplots(4, 8, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < 32:
            ax.imshow(first_conv_activation[:, :, i], cmap='viridis')
        ax.axis('off')

    plt.suptitle(f'Feature Maps After Conv1 (Input: {class_names[y_test[0]]})', fontsize=14)
    plt.tight_layout()
    plt.show()

    print("Each feature map shows what one filter 'sees' in the image.")
    print("Bright areas indicate strong pattern matches.")
except Exception as e:
    print(f"Feature map visualization skipped ({type(e).__name__}: {e})")
    print("This visualization requires a fully-built model. On Colab, run all cells in order.")

## Part 7: CNN vs Dense Network Comparison

In [ ]:
# Build dense (MLP) model with similar parameter count
dense_model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

dense_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("=== Dense Network Architecture ===")
dense_model.summary()

In [ ]:
# Train dense model
print("\nTraining Dense Network...")
dense_history = dense_model.fit(
    X_train_flat, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Compare results
dense_test_loss, dense_test_acc = dense_model.evaluate(X_test_flat, y_test, verbose=0)

print("\n=== Model Comparison ===")
print(f"{'Model':<15} {'Test Accuracy':<15} {'Parameters':<15}")
print("-" * 45)
print(f"{'CNN':<15} {test_acc:<15.1%} {cnn_model.count_params():<15,}")
print(f"{'Dense':<15} {dense_test_acc:<15.1%} {dense_model.count_params():<15,}")
print(f"\nCNN advantage: {(test_acc - dense_test_acc)*100:+.1f} percentage points")

print("\n💡 CNNs achieve better accuracy with fewer parameters!")
print("   Spatial structure helps CNNs learn efficiently.")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(8, 5))

models = ['Dense\nNetwork', 'CNN']
accuracies = [dense_test_acc, test_acc]
params = [dense_model.count_params(), cnn_model.count_params()]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#22c55e')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.8, 1.0)

ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, [p/1000 for p in params], width, label='Parameters (K)', color='#f97316')
ax2.set_ylabel('Parameters (thousands)')

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_title('CNN vs Dense Network on Fashion-MNIST')

ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Part 8: TODO - Experiment with Regularization

Try adding dropout and batch normalization to reduce overfitting.

In [ ]:
# TODO: Build a regularized CNN
# Hint: Add Dropout layers after Conv2D and Dense layers
# Hint: Add BatchNormalization after Conv2D layers

regularized_model = keras.Sequential([
    # First conv block with batch norm and dropout
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Second conv block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Classification head
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

regularized_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("=== Regularized CNN ===")
print("Added: BatchNormalization, Dropout(0.25, 0.5)")
print(f"Parameters: {regularized_model.count_params():,}")

In [ ]:
# TODO: Train the regularized model and compare
# Uncomment and run:

# print("Training regularized CNN...")
# reg_history = regularized_model.fit(
#     X_train_cnn, y_train,
#     epochs=5,
#     batch_size=64,
#     validation_split=0.1,
#     verbose=1
# )
# reg_test_loss, reg_test_acc = regularized_model.evaluate(X_test_cnn, y_test, verbose=0)
# print(f"\nRegularized CNN Test Accuracy: {reg_test_acc:.1%}")
# print(f"Original CNN Test Accuracy: {test_acc:.1%}")

## Part 9: When to Use CNNs

### Decision Heuristic

| Data Type | Best Approach |
|-----------|---------------|
| **Images** | CNN |
| **Tabular data** | Gradient boosting (XGBoost, LightGBM) |
| **Text** | Transformers or RNNs |
| **Time series** | CNN-1D or RNNs |
| **Small datasets (<1K)** | Simpler models, pretrained CNNs |

### CNN Strengths
- Spatial structure (images, spectrograms)
- Translation invariance (object can be anywhere)
- Parameter efficiency (weight sharing)

### CNN Limitations
- Needs enough data (thousands of examples)
- Computational cost for training
- Less interpretable than simpler models

In [ ]:
print("=== When to Use CNNs ===")
print("")
print("✓ USE CNNs when:")
print("  - Data has spatial structure (images, spectrograms)")
print("  - You have thousands of labeled examples")
print("  - Translation invariance matters (object position varies)")
print("")
print("✗ AVOID CNNs when:")
print("  - Data is tabular (use gradient boosting)")
print("  - Dataset is small (<1000 examples)")
print("  - Interpretability is critical")
print("")
print("💡 For small image datasets, use transfer learning with pretrained CNNs!")

## Self-Check

In [ ]:
# SELF-CHECK: Verify your CNN works correctly
assert test_acc > 0.85, f"CNN accuracy should be >85%, got {test_acc:.1%}"
assert test_acc > dense_test_acc, "CNN should outperform dense network on images"
assert cnn_model.count_params() < dense_model.count_params(), "CNN should have fewer parameters"
print(f"✅ Self-check passed!")
print(f"   CNN accuracy: {test_acc:.1%}")
print(f"   Dense accuracy: {dense_test_acc:.1%}")
print(f"   CNN parameters: {cnn_model.count_params():,}")
print(f"   Dense parameters: {dense_model.count_params():,}")

## Part 10: Stakeholder Summary

### TODO: Write a 3-bullet summary (~100 words) for the PM

Template:
- **What CNNs do:** CNNs are specialized neural networks for [images/spatial data] that learn to detect [patterns like edges and textures] automatically.
- **Why they work:** By preserving spatial structure and using [weight sharing], CNNs achieve [higher accuracy with fewer parameters] than dense networks.
- **When to use:** Use CNNs for [image classification, object detection] when you have [enough labeled data]. For small datasets, consider [transfer learning from pretrained models].

### Your Summary:

*Write your explanation here...*

---

## Key Takeaways

1. **CNNs exploit spatial structure** by sliding filters over images
2. **Convolution filters** detect local patterns (edges, textures, shapes)
3. **Pooling** reduces dimensions while preserving important features
4. **CNNs beat dense networks** on image data with fewer parameters
5. **Regularization** (dropout, batch norm) helps prevent overfitting
6. **Use CNNs for images**, not tabular data

### Next Steps
- Explore the interactive playground
- Complete the quiz
- Move to Module 17: Transformers